# 拆开一个两层 Transformer

用六个张量实验观察词嵌入、因果掩码、两个注意力头、旋转位置编码、前馈层与残差。各单元附有 CPU 运行结果，模型训练和生成见同目录的 [demo.py](demo.py)。

In [1]:
import torch
import torch.nn.functional as F

from attention import single_head_attention, multi_head_attention, apply_rope, multi_head_attention_rope
from layers import swiglu_ffn, rms_norm, transformer_block

## 1. 词嵌入和下一个 token

输入取每行的前四个 ID，预测目标取后四个。词表有 260 项，每个位置都会得到 260 个分数。

In [2]:
torch.manual_seed(0)

V = 260   # 词表大小
d = 8     # 隐藏向量长度

# 两条示例序列，每条有 5 个 token。
# 这些数字只用于演示，不对应特定文本。
batch_ids = torch.tensor(
    [
        [10, 20, 30, 40, 50],
        [60, 70, 80, 90, 100],
    ],
    dtype=torch.long,
)

# 前 4 个 token 作为输入，后 4 个 token 作为目标。
input_ids = batch_ids[:, :-1]
target_ids = batch_ids[:, 1:]

embedding = torch.nn.Embedding(V, d)
h = embedding(input_ids)

print("输入形状：", input_ids.shape)
print("目标形状：", target_ids.shape)
print("Embedding 权重形状：", embedding.weight.shape)
print("隐藏表示形状：", h.shape)

# 完整模型中，这里的 h 应当来自：
# Embedding → 多个 Transformer block → 最终归一化。
#
# 本实验暂时直接使用 Embedding 输出，只验证输出端的接口。

W_vocab = torch.randn(d, V, requires_grad=True)

logits_demo = h @ W_vocab

# 把每个位置的 V 个分数转换成概率。
probs_demo = torch.softmax(logits_demo, dim=-1)

print("分数形状：", logits_demo.shape)
print("概率形状：", probs_demo.shape)
print("各位置的概率总和：")
print(probs_demo.sum(dim=-1))

loss_demo = F.cross_entropy(
    logits_demo.reshape(-1, V),
    target_ids.reshape(-1),
)

print("损失形状：", loss_demo.shape)
print("损失数值：", loss_demo.item())

输入形状： torch.Size([2, 4])
目标形状： torch.Size([2, 4])
Embedding 权重形状： torch.Size([260, 8])
隐藏表示形状： torch.Size([2, 4, 8])
分数形状： torch.Size([2, 4, 260])
概率形状： torch.Size([2, 4, 260])
各位置的概率总和：
tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000]], grad_fn=<SumBackward1>)
损失形状： torch.Size([])
损失数值： 6.738194465637207


## 2. 单头因果掩码

修改句末的输入，检查前面各位置的输出是否保持一致。

In [3]:
torch.manual_seed(0)
B, T, d = 2, 4, 8
d_k = d
d_v = d

h = torch.randn(B, T, d)

# 用较小的随机值初始化，便于观察本例的注意力权重。
# requires_grad_() 开启对这些权重的梯度追踪。
W_q = (0.1 * torch.randn(d, d_k)).requires_grad_()
W_k = (0.1 * torch.randn(d, d_k)).requires_grad_()
W_v = (0.1 * torch.randn(d, d_v)).requires_grad_()
W_o = (0.1 * torch.randn(d_v, d)).requires_grad_()


# ---------- 执行一次前向计算 ----------

out, weights = single_head_attention(
    h, W_q, W_k, W_v, W_o
)

print("输出形状：", out.shape)
print("注意力权重形状：", weights.shape)

print("\n第一条序列的注意力权重：")
print(weights[0])

print("\n每个查询位置的权重总和：")
print(weights.sum(dim=-1))

# 复制输入，再大幅修改最后一个位置的向量。
h_changed = h.clone()
h_changed[:, -1, :] += 100

out_changed, _ = single_head_attention(
    h_changed, W_q, W_k, W_v, W_o
)

# 比较最后一个位置之前的所有输出。
print(
    torch.allclose(
        out[:, :-1, :],
        out_changed[:, :-1, :],
        atol=1e-6,
    )
)

输出形状： torch.Size([2, 4, 8])
注意力权重形状： torch.Size([2, 4, 4])

第一条序列的注意力权重：
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.4903, 0.5097, 0.0000, 0.0000],
        [0.3538, 0.3302, 0.3160, 0.0000],
        [0.2500, 0.2483, 0.2524, 0.2492]], grad_fn=<SelectBackward0>)

每个查询位置的权重总和：
tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000]], grad_fn=<SumBackward1>)
True


## 3. 多头注意力

两个头各自得到一张 4 × 4 权重表。比较它们的数值，再检查每行的和。

In [4]:
torch.manual_seed(0)
B, T, d = 2, 4, 8
num_heads = 2

h = torch.randn(B, T, d)

W_q = (0.1 * torch.randn(d, d)).requires_grad_()
W_k = (0.1 * torch.randn(d, d)).requires_grad_()
W_v = (0.1 * torch.randn(d, d)).requires_grad_()
W_o = (0.1 * torch.randn(d, d)).requires_grad_()


# ---------- 执行前向计算 ----------

out, weights = multi_head_attention(
    h, W_q, W_k, W_v, W_o, num_heads
)

print("输出形状：", out.shape)
print("注意力权重形状：", weights.shape)

print("\n第一条序列，头 0 的权重：")
print(weights[0, 0])

print("\n第一条序列，头 1 的权重：")
print(weights[0, 1])

print("\n每个头、每个查询位置的权重总和：")
print(weights.sum(dim=-1))

输出形状： torch.Size([2, 4, 8])
注意力权重形状： torch.Size([2, 2, 4, 4])

第一条序列，头 0 的权重：
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5043, 0.4957, 0.0000, 0.0000],
        [0.3545, 0.3231, 0.3224, 0.0000],
        [0.2512, 0.2474, 0.2498, 0.2516]], grad_fn=<SelectBackward0>)

第一条序列，头 1 的权重：
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.4819, 0.5181, 0.0000, 0.0000],
        [0.3411, 0.3393, 0.3196, 0.0000],
        [0.2489, 0.2502, 0.2537, 0.2473]], grad_fn=<SelectBackward0>)

每个头、每个查询位置的权重总和：
tensor([[[1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000]]], grad_fn=<SumBackward1>)


## 4. 旋转位置编码

令各位置的输入相同。把每对特征按位置旋转，可以直接观察四个位置的结果和各自的向量长度。

In [5]:
# 1 条序列，1 个头，4 个位置，每个向量有 4 个特征。
x_demo = torch.zeros(1, 1, 4, 4)

# 把下标 0、2 的分量设为 1。
# 因此每个位置的向量都是 [1, 0, 1, 0]。
x_demo[..., 0::2] = 1.0

x_rotated = apply_rope(x_demo)

print("旋转前：")
print(x_demo[0, 0])

print("\n旋转后：")
print(x_rotated[0, 0])

assert torch.allclose(x_rotated.norm(dim=-1), x_demo.norm(dim=-1))

旋转前：
tensor([[1., 0., 1., 0.],
        [1., 0., 1., 0.],
        [1., 0., 1., 0.],
        [1., 0., 1., 0.]])

旋转后：
tensor([[ 1.0000,  0.0000,  1.0000,  0.0000],
        [ 0.5403,  0.8415,  0.9999,  0.0100],
        [-0.4161,  0.9093,  0.9998,  0.0200],
        [-0.9900,  0.1411,  0.9996,  0.0300]])


In [6]:
torch.manual_seed(0)

B, T, d = 2, 4, 8
num_heads = 2

h = torch.randn(B, T, d)

W_q = (0.1 * torch.randn(d, d)).requires_grad_()
W_k = (0.1 * torch.randn(d, d)).requires_grad_()
W_v = (0.1 * torch.randn(d, d)).requires_grad_()
W_o = (0.1 * torch.randn(d, d)).requires_grad_()

out, weights = multi_head_attention_rope(
    h, W_q, W_k, W_v, W_o, num_heads
)

print("输出形状：", out.shape)
print("注意力权重形状：", weights.shape)

输出形状： torch.Size([2, 4, 8])
注意力权重形状： torch.Size([2, 2, 4, 4])


## 5. SwiGLU 前馈层

每个位置用同一组参数独立计算，改变最后一个位置不会改变前面的位置。

In [7]:
torch.manual_seed(0)
B, T, d = 2, 4, 8
d_ff = 16

h = torch.randn(B, T, d)

W_gate = (
    0.1 * torch.randn(d, d_ff)
).requires_grad_()

W_up = (
    0.1 * torch.randn(d, d_ff)
).requires_grad_()

W_down = (
    0.1 * torch.randn(d_ff, d)
).requires_grad_()


# ---------- 前向计算 ----------

out = swiglu_ffn(h, W_gate, W_up, W_down)

print("输入形状：", h.shape)
print("门控分支投影形状：", (h @ W_gate).shape)
print("特征分支投影形状：", (h @ W_up).shape)
print("输出形状：", out.shape)

h_changed = h.clone()

# 只修改最后一个位置的输入。
h_changed[:, -1, :] += 100

out_changed = swiglu_ffn(
    h_changed, W_gate, W_up, W_down
)

# 比较前面所有位置的输出。
print(
    torch.allclose(
        out[:, :-1, :],
        out_changed[:, :-1, :],
        atol=1e-6,
    )
)

输入形状： torch.Size([2, 4, 8])
门控分支投影形状： torch.Size([2, 4, 16])
特征分支投影形状： torch.Size([2, 4, 16])
输出形状： torch.Size([2, 4, 8])
True


## 6. RMSNorm、残差和反向传播

先观察一个归一化例子，再把注意力和前馈层放入同一个残差块，检查梯度能否传回输入及门控权重。

In [8]:
example = torch.tensor([
    [
        [3.0, 4.0],
        [30.0, 40.0],
    ]
])

weight = torch.ones(2)

print(rms_norm(example, weight))

tensor([[[0.8485, 1.1314],
         [0.8485, 1.1314]]])


In [9]:
torch.manual_seed(0)

B, T, d = 2, 4, 8
d_ff = 16
num_heads = 2

# 演示中也追踪输入的梯度。
h = torch.randn(B, T, d, requires_grad=True)

params = {
    # 两套独立的 RMSNorm 参数。
    "norm1_weight": torch.ones(
        d, requires_grad=True
    ),
    "norm2_weight": torch.ones(
        d, requires_grad=True
    ),

    # 注意力参数。
    "W_q": (
        0.1 * torch.randn(d, d)
    ).requires_grad_(),

    "W_k": (
        0.1 * torch.randn(d, d)
    ).requires_grad_(),

    "W_v": (
        0.1 * torch.randn(d, d)
    ).requires_grad_(),

    "W_o": (
        0.1 * torch.randn(d, d)
    ).requires_grad_(),

    # SwiGLU 参数。
    "W_gate": (
        0.1 * torch.randn(d, d_ff)
    ).requires_grad_(),

    "W_up": (
        0.1 * torch.randn(d, d_ff)
    ).requires_grad_(),

    "W_down": (
        0.1 * torch.randn(d_ff, d)
    ).requires_grad_(),
}

block_out, weights = transformer_block(
    h, params, num_heads
)

print("输入形状：", h.shape)
print("block 输出形状：", block_out.shape)
print("注意力权重形状：", weights.shape)

# 仅用于观察反向传播的示例损失。
loss_demo = block_out.square().mean()

loss_demo.backward()

print("输入梯度形状：", h.grad.shape)
print(
    "第一套归一化参数的梯度形状：",
    params["norm1_weight"].grad.shape,
)
print(
    "FFN 门控权重的梯度形状：",
    params["W_gate"].grad.shape,
)

输入形状： torch.Size([2, 4, 8])
block 输出形状： torch.Size([2, 4, 8])
注意力权重形状： torch.Size([2, 2, 4, 4])
输入梯度形状： torch.Size([2, 4, 8])
第一套归一化参数的梯度形状： torch.Size([8])
FFN 门控权重的梯度形状： torch.Size([8, 16])
